# Challenge 3: Robust Tagging Under Changing Detector Conditions

## What's the challenge?

A particle physics detector maps candidate particles in **η (eta) and φ (phi)** — pseudorapidity and azimuthal angle. When detector elements go offline — dead channels, noisy modules, alignment failures — candidates in those regions disappear from the event.

Your task: **train a model whose anomaly detection doesn't fall apart when parts of the detector go dead.**

We simulate this by zeroing out PF candidates whose (η, φ) falls inside a dead region of the detector, using the same convention as event padding (all features set to 0, pt = 0). Your model is never told which region is dead or how large it is.

---

## Training data

- **Background only**: QCD, Drell-Yan, tt̄, W+jets — what the detector sees most of the time
- Each event is a fixed-size tensor of PF candidates: `[N, 7]` with features `(pt, η, φ, dxy, dxy_sig, is_pf, pdgId)`. Padding rows have `pt == 0`.

## Scoring

At eval time your model sees both background and signal events. It outputs a per-event anomaly score. Scores are evaluated against true labels via **AUC vs. degradation severity**. A robust model maintains high AUC as more of the detector goes dark. The leaderboard score is the area under that AUC-vs-severity curve.

---

## How to use this notebook

1. **Edit `degrade`** (Section 2) — controls how dead detector regions are simulated during training.
2. **Edit `Model`** (Section 3) — `fit()` trains on background data, `predict()` writes anomaly scores.
3. **Run Train** (Section 4) and **Eval** (Section 5) to check performance.
4. **Submit** (Section 6) — paste your `degrade` function and `Model` class into `model.py` and upload to Codabench.

The defaults below are a working baseline.

---
## Section 1: Setup

In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

from embedding.models import TransformerEncoder, Projector
from embedding.preprocs import PFPreProcessor
from embedding.loss import SupConLoss
from embedding.training import make_train_val_split, build_train_val_loaders
from embedding.utils.data_utils import load_data

DATA_DIR = "REPLACE_ME"   # directory containing train + eval files
OUT_DIR  = "REPLACE_ME"   # where predictions are written

os.makedirs(OUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

---
## Section 2: Your `degrade` function

This is the **detector degradation simulator** used during training. It takes a batch of events and returns the same batch with some candidates zeroed out — simulating dead detector cells.

**Interface:** `degrade(x: Tensor[B, N, F]) -> Tensor[B, N, F]`
- `x[..., 0]` is pt — candidates with `pt == 0` are already padding, leave them
- `x[..., 1]` is η, `x[..., 2]` is φ
- Zero all features of dead candidates (same convention as padding)

**Some things to think about:**
- The η-φ plane can be discretised into a regular grid of cells. A dead region is one or more cells where candidates go missing.
- Dead cells don't have to kill every candidate inside them — in a real detector, a partially noisy module might drop candidates stochastically.
- During training, the dead region should vary per event so the model can't memorise a fixed pattern.
- The distribution of severities (how much of the detector is dead) you train on matters. Think about how it relates to what gets tested at eval.
- Multiple disconnected dead regions are more realistic than one large contiguous patch.

The default below is a minimal working baseline — a single random dead patch per event. You should be able to do better.

In [ ]:
# =====================================================================
# YOUR degrade FUNCTION — edit this
# =====================================================================

def degrade(x: torch.Tensor) -> torch.Tensor:
    """Baseline: one random dead patch per event."""
    x = x.clone()
    B = x.size(0)
    eta_c = torch.empty(B, 1, device=x.device).uniform_(-2.0, 2.0)
    phi_c = torch.empty(B, 1, device=x.device).uniform_(-torch.pi, torch.pi)
    deta  = torch.empty(B, 1, device=x.device).uniform_(0.2, 1.5)
    dphi  = torch.empty(B, 1, device=x.device).uniform_(0.2, 1.5)
    eta, phi = x[..., 1], x[..., 2]
    dead = (
        (eta >= eta_c - deta / 2) & (eta < eta_c + deta / 2) &
        (phi >= phi_c - dphi / 2) & (phi < phi_c + dphi / 2) &
        (x[..., 0] > 0)
    )
    x[dead] = 0.0
    return x

# =====================================================================

---
## Section 3: Your `Model` class

Implement the submission interface:

- `fit()` — train on `DATA_DIR/train.pt` (background events only)
- `predict()` — write anomaly scores to `OUT_DIR` as `[E, 2]` arrays where column 1 is the signal/anomaly probability

Your `fit()` should use your `degrade` function above to augment events during training. The default baseline uses a contrastive objective (SupConLoss) pairing nominal and degraded views of each event.

In [ ]:
# =====================================================================
# YOUR Model CLASS — implement / modify this
# =====================================================================

NUM_BG_CLASSES = 4  # QCD, DY, TT, WJets

class Model:
    def __init__(self, data_dir, out_dir):
        self.data_dir = data_dir
        self.out_dir  = out_dir

        self.preproc    = PFPreProcessor(norm_constants={}).to(device)
        self.encoder    = TransformerEncoder(
            num_features=self.preproc.num_features,
            embed_size=128, latent_dim=6, num_heads=8, num_layers=4,
        ).to(device)
        self.projector  = Projector(6, 12, hidden_dim=48).to(device)
        self.classifier = nn.Linear(12, NUM_BG_CLASSES).to(device)

    @staticmethod
    def _cls_mask(x):
        m = x[..., 0] == 0
        return torch.cat([torch.zeros(m.size(0), 1, device=m.device, dtype=torch.bool), m], dim=1)

    def _embed(self, x):
        return F.normalize(self.projector(self.encoder(self.preproc(x), None, self._cls_mask(x))), dim=1)

    def fit(self):
        feature_block, label_block = load_data(
            os.path.join(self.data_dir, "REPLACE_ME"),  # train file
            map_location="cpu",
        )
        X_tr, y_tr, X_val, y_val, _, _ = make_train_val_split(feature_block, label_block, val_size=0.1)
        train_loader, val_loader = build_train_val_loaders(
            X_tr, y_tr, X_val, y_val, device=device, batch_size=256, pfcands=True
        )

        criterion  = SupConLoss(temperature=0.07)
        ce_loss_fn = nn.CrossEntropyLoss()
        optimizer  = torch.optim.Adam(
            list(self.preproc.parameters()) + list(self.encoder.parameters())
            + list(self.projector.parameters()) + list(self.classifier.parameters()),
            lr=1e-3,
        )

        for epoch in range(10):
            self.preproc.train(); self.encoder.train()
            self.projector.train(); self.classifier.train()
            for x, _, labels in train_loader:
                x, labels = x.to(device), labels.to(device)
                x_aug = degrade(x)

                optimizer.zero_grad()
                emb_in  = self._embed(x)
                emb_aug = self._embed(x_aug)
                features = torch.stack([emb_in, emb_aug], dim=1)  # [B, 2, proj_dim]
                loss = criterion(features, labels) + ce_loss_fn(self.classifier(emb_in), labels)
                loss.backward()
                optimizer.step()

            self.preproc.eval(); self.encoder.eval()
            self.projector.eval(); self.classifier.eval()
            val_loss = val_correct = 0
            with torch.no_grad():
                for x, _, labels in val_loader:
                    x, labels = x.to(device), labels.to(device)
                    emb    = self._embed(x)
                    logits = self.classifier(emb)
                    val_loss    += ce_loss_fn(logits, labels).item() * x.size(0)
                    val_correct += (logits.argmax(1) == labels).float().sum().item()
            N = len(val_loader.dataset)
            print(f"epoch {epoch+1}/10  val_loss={val_loss/N:.4f}  val_acc={val_correct/N:.4f}")

    @torch.no_grad()
    def _anomaly_scores(self, path):
        features = torch.load(path, map_location=device)
        self.preproc.eval(); self.encoder.eval(); self.projector.eval(); self.classifier.eval()
        emb      = self._embed(features)
        bg_probs = torch.softmax(self.classifier(emb), dim=1)
        anomaly  = 1.0 - bg_probs.max(dim=1).values
        return torch.stack([1.0 - anomaly, anomaly], dim=1).cpu().numpy()

    def predict(self):
        np.save(os.path.join(self.out_dir, "pred_nominal.npy"),
                self._anomaly_scores(os.path.join(self.data_dir, "REPLACE_ME")))  # eval nominal (public)

        for path in glob.glob(os.path.join(self.data_dir, "REPLACE_ME", "*.pt")):  # eval degraded (public)
            sev = re.search(r"\d+", os.path.basename(path)).group()
            np.save(os.path.join(self.out_dir, f"pred_severity_{sev}.npy"), self._anomaly_scores(path))

# =====================================================================

print("Model class defined.")

---
## Section 4: Train

Runs `fit()` on the background training data. This is the same thing that happens in the final submission — no hidden steps.

In [ ]:
model = Model(DATA_DIR, OUT_DIR)
model.fit()

---
## Section 5: Evaluate

Runs `predict()` on the held-out eval events (backgrounds + signals, with and without degradation) and plots the AUC vs. severity curve. A flat curve near AUC = 1 is what you're aiming for.

In [ ]:
model.predict()

labels = np.load(os.path.join(DATA_DIR, "REPLACE_ME"))  # labels.npy (public eval)

def auc_from(pred_path):
    probs = np.load(pred_path)
    return roc_auc_score(labels, probs[:, 1])

severities = [0]
aucs       = [auc_from(os.path.join(OUT_DIR, "pred_nominal.npy"))]

sev_files = sorted(
    glob.glob(os.path.join(OUT_DIR, "pred_severity_*.npy")),
    key=lambda p: int(re.search(r"\d+", os.path.basename(p)).group())
)
for path in sev_files:
    sev = int(re.search(r"\d+", os.path.basename(path)).group())
    severities.append(sev)
    aucs.append(auc_from(path))

robustness_score = np.trapz(aucs, severities) / (severities[-1] - severities[0]) if len(severities) > 1 else aucs[0]

print(f"\nRobustness score (area under AUC-vs-severity): {robustness_score:.4f}")
print(f"{'Severity':>10}  {'AUC':>8}")
for s, a in zip(severities, aucs):
    print(f"{s:>10}%  {a:>8.4f}")

plt.figure(figsize=(7, 5))
plt.plot(severities, aucs, marker="o", linewidth=2)
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="random")
plt.xlabel("Dead detector fraction (% of η-φ plane)")
plt.ylabel("AUC (signal vs background)")
plt.title(f"Robustness score: {robustness_score:.4f}")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 6: Submit to Codabench

Your submission is a single `model.py` file. Copy your `degrade` function (Section 2) and your `Model` class (Section 3) into it — the rest of the file (imports, `device`, `NUM_BG_CLASSES`) is already there.

Download `model.py` from the competition page, paste in your code, and upload it as your submission. Codabench will call `Model(data_dir, out_dir).fit()` then `.predict()` and score the outputs automatically.